In [ ]:
# === 노트북 공통 preamble (llm_math 패키지 로드) ===
import sys
from pathlib import Path

# llm_math 패키지를 찾을 수 있는 후보 경로들
_candidates = [
    '.', 'src', '..', '../src',
    '/content/llm-math-book/src',
    '/content/llm-math-book',
    '/workspace/src',
    '/workspace',
]
# 현재 디렉토리의 상위 디렉토리들도 후보에 추가 (notebooks/ 폴더에서 실행 시)
for p in Path.cwd().parents:
    _candidates.append(str(p / 'src'))
    _candidates.append(str(p))

for p in _candidates:
    if p and p not in sys.path and Path(p).exists():
        sys.path.insert(0, p)

# llm_math import 시도
try:
    from llm_math import viz, bench, data
    _LLM_MATH_OK = True
except ImportError as e:
    _LLM_MATH_OK = False
    print(f"[주의] llm_math 패키지 로드 실패: {e}")
    print("  GitHub 레포를 클론하고 colab_setup.sh를 실행하세요.")
# === preamble 끝 ===


# Ch 17. 트랜스포머 아키텍처 — 블록을 조립하다

> **학습 목표**
> - "Attention is All You Need" 원 논문의 아키텍처를 블록별로 이해한다
> - Pre-LN vs Post-LN의 차이와 학습 안정성을 설명한다
> - 인코더-디코더, 디코더 온리, 인코더 온리의 용도를 비교한다

## 17.1 트랜스포머의 탄생

2017년 Vaswani et al. — RNN 없이 어텐션만으로 시퀀스 처리.

이점:
1. **병렬화**: RNN의 순차 처리 제거
2. **장기 의존성**: 모든 토큰 직접 연결
3. **확장성**: 모델 크기 키우기 쉬움 → LLM 시대의 기반

## 17.2 세 가지 변형

| 변형 | 구조 | 사용 모델 |
|---|---|---|
| 인코더-디코더 | 둘 다 | 기계번역 (원래 트랜스포머, T5, BART) |
| 인코더 온리 | 양방향 어텐션 | BERT, RoBERTa |
| 디코더 온리 | 인과적 어텐션 | GPT, LLaMA, Qwen |

LLM 시대는 **디코더 온리**가 지배적. 다음 토큰 예측만으로 강력한 능력 학습.

## 17.3 트랜스포머 블록 구조

각 블록은:
1. Multi-Head Self-Attention
2. 잔차 연결 (Residual)
3. 층 정규화 (LayerNorm/RMSNorm)
4. Feed-Forward Network (FFN)
5. 잔차 연결
6. 정규화

**Post-LN** (원래):
$$\mathbf{x}' = \mathrm{LayerNorm}(\mathbf{x} + \mathrm{Attn}(\mathbf{x}))$$

**Pre-LN** (GPT-2 이후 표준):
$$\mathbf{x}' = \mathbf{x} + \mathrm{Attn}(\mathrm{LayerNorm}(\mathbf{x}))$$

Pre-LN이 깊은 모델에서 학습 더 안정적.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

class TransformerBlock(nn.Module):
    """Pre-LN Transformer Block."""
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1, pre_ln=True):
        super().__init__()
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        self.W_qkv = nn.Linear(d_model, 3 * d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
        )
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
        self.pre_ln = pre_ln

    def attention(self, x, mask=None):
        B, T, D = x.shape
        qkv = self.W_qkv(x)
        q, k, v = qkv.chunk(3, dim=-1)
        q = q.view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        scores = q @ k.transpose(-1, -2) / (self.d_k ** 0.5)
        if mask is not None:
            scores = scores + mask
        weights = F.softmax(scores, dim=-1)
        out = (weights @ v).transpose(1, 2).contiguous().view(B, T, D)
        return self.W_o(out)

    def forward(self, x, mask=None):
        if self.pre_ln:
            # Pre-LN
            x = x + self.dropout(self.attention(self.ln1(x), mask))
            x = x + self.dropout(self.ffn(self.ln2(x)))
        else:
            # Post-LN
            x = self.ln1(x + self.dropout(self.attention(x, mask)))
            x = self.ln2(x + self.dropout(self.ffn(x)))
        return x

# Test
d_model, n_heads, d_ff = 64, 8, 256
block = TransformerBlock(d_model, n_heads, d_ff)
x = torch.randn(2, 10, d_model)
out = block(x)
print(f"입력: {x.shape}")
print(f"Output: {out.shape}")
print(f"Block Parameter Count: {sum(p.numel() for p in block.parameters()):,}")


## 17.4 Position-wise Feed-Forward Network (FFN)

$$\mathrm{FFN}(\mathbf{x}) = \mathrm{GELU}(\mathbf{x} W_1 + \mathbf{b}_1) W_2 + \mathbf{b}_2$$

- $W_1 \in \mathbb{R}^{d \times d_{ff}}$, $W_2 \in \mathbb{R}^{d_{ff} \times d}$
- 보통 $d_{ff} = 4d$ (예: $d=512, d_{ff}=2048$)
- 각 위치마다 **독립적으로** 적용 (position-wise)

**SwiGLU** (LLaMA): GLU 활성화로 성능 향상
$$\mathrm{SwiGLU}(\mathbf{x}) = \mathrm{SiLU}(\mathbf{x} W_1) \odot (\mathbf{x} W_3) W_2$$


In [ ]:
# FFN 변형 비교
class StandardFFN(nn.Module):
    def __init__(self, d, d_ff):
        super().__init__()
        self.w1 = nn.Linear(d, d_ff)
        self.w2 = nn.Linear(d_ff, d)

    def forward(self, x):
        return self.w2(F.gelu(self.w1(x)))

class SwiGLUFFN(nn.Module):
    def __init__(self, d, d_ff):
        super().__init__()
        self.w1 = nn.Linear(d, d_ff, bias=False)
        self.w2 = nn.Linear(d_ff, d, bias=False)
        self.w3 = nn.Linear(d, d_ff, bias=False)

    def forward(self, x):
        return self.w2(F.silu(self.w1(x)) * self.w3(x))

d, d_ff = 64, 256
std_ffn = StandardFFN(d, d_ff)
glu_ffn = SwiGLUFFN(d, d_ff)
print(f"Standard FFN params: {sum(p.numel() for p in std_ffn.parameters()):,}")
print(f"SwiGLU FFN params: {sum(p.numel() for p in glu_ffn.parameters()):,}")

x = torch.randn(2, 10, d)
y1 = std_ffn(x)
y2 = glu_ffn(x)
print(f"Standard FFN Output: {y1.shape}")
print(f"SwiGLU FFN Output: {y2.shape}")
print("\n=> SwiGLU는 W3 추가로 파라미터 더 많지만, Performance Improvement. LLaMA Standard.")


## 17.5 잔차 연결과 정규화의 역할

**잔차 연결**: $\mathbf{x}_{\ell+1} = \mathbf{x}_\ell + f_\ell(\mathbf{x}_\ell)$

- 그래디언트 직접 경로 제공 → 깊은 모델 학습 가능
- 정보 병목 방지

**정규화**: 활성값 분포 안정화 → 더 큰 학습률, 더 빠른 수렴


In [ ]:
# 잔차 연결 효과: 깊이에 따른 그래디언트 흐름
def test_grad_flow(depth, use_residual):
    """Depth별 그래디언트 흐름 시뮬레이션."""
    d = 64
    layers = [nn.Linear(d, d) for _ in range(depth)]
    x = torch.randn(1, d, requires_grad=True)
    target = torch.randn(1, d)
    loss_fn = nn.MSELoss()

    out = x
    for layer in layers:
        if use_residual:
            out = out + layer(out)
        else:
            out = layer(out)
    loss = loss_fn(out, target)
    loss.backward()
    return x.grad.norm().item()

print(f"{'depth':>6} | {'잔차 X':>15} | {'잔차 O':>15}")
print("-" * 40)
for d in [5, 10, 20, 50]:
    g_no = test_grad_flow(d, False)
    g_yes = test_grad_flow(d, True)
    print(f"{d:>6} | {g_no:>15.6e} | {g_yes:>15.6e}")
print("\n=> 잔차 연결이 있으면 깊어져도 그래디언트가 잘 흐른다.")


## 17.6 모델 파라미터 수 계산

트랜스포머 블록당:
- QKV 프로젝션: $3 d^2$
- 출력 프로젝션: $d^2$
- FFN (standard): $2 \cdot 4d^2 = 8d^2$
- LayerNorm: $4d$
- **총**: $\approx 12 d^2$ per block

전체 모델 ($L$ 블록):
- 임베딩: $|V| \cdot d$
- 블록: $L \cdot 12 d^2$
- 출력층: $|V| \cdot d$ (weight tying 시 생략)

대략 $P \approx 12 L d^2$ (임베딩 무시 시).


In [ ]:
# 파라미터 수 계산
def count_transformer_params(vocab_size, d_model, n_layers, d_ff=None, tie_weights=True):
    if d_ff is None: d_ff = 4 * d_model
    # 임베딩
    emb = vocab_size * d_model
    # 블록당
    qkv = 3 * d_model * d_model
    out_proj = d_model * d_model
    ffn = 2 * d_model * d_ff
    ln = 4 * d_model  # 두 개 LN
    block = qkv + out_proj + ffn + ln
    # 출력층
    output = 0 if tie_weights else vocab_size * d_model
    total = emb + n_layers * block + output
    return total, {'emb': emb, 'block': block, 'n_layers': n_layers, 'output': output}

# 다양한 Magnitude Comparison
print(f"{'Model':>12} | {'V':>8} | {'d':>5} | {'L':>3} | {'Params':>12}")
print("-" * 50)
for name, V, d, L in [
    ('Mini', 1000, 64, 4),
    ('Small', 10000, 256, 6),
    ('GPT-1', 40000, 768, 12),
    ('GPT-2', 50257, 768, 12),
    ('GPT-2 medium', 50257, 1024, 24),
    ('GPT-3 6.7B', 50257, 4096, 32),
    ('LLaMA-7B', 32000, 4096, 32),
]:
    n, _ = count_transformer_params(V, d, L)
    print(f"{name:>12} | {V:>8} | {d:>5} | {L:>3} | {n:>12,}")


## 17.7 핵심 정리

| 구성 요소 | 역할 |
|---|---|
| Multi-Head Attention | 토큰 간 관계 학습 |
| FFN | 비선형 변환 (position-wise) |
| 잔차 연결 | 깊은 모델 학습 가능 |
| LayerNorm/RMSNorm | 활성값 안정화 |
| Positional Encoding | 순서 정보 주입 |

| 변형 | 용도 |
|---|---|
| Post-LN | 원래 트랜스포머 |
| Pre-LN | GPT-2+ 표준 |
| SwiGLU FFN | LLaMA 표준 |

## 연습문제

1. Pre-LN과 Post-LN 블록을 같은 깊이로 학습하고 안정성을 비교하라.
2. Standard FFN과 SwiGLU FFN을 같은 파라미터 수로 비교하라 (d_ff 조절 필요).
3. 잔차 연결이 있을 때와 없을 때 깊이 10, 20, 50의 학습을 시도하고 그래디언트를 비교하라.
4. GPT-2 small (d=768, L=12, V=50257)의 파라미터 수를 계산하라.
5. 트랜스포머 블록에서 어텐션과 FFN의 파라미터 비율을 계산하라.

> 해설: `solutions/ch17_solutions.ipynb`
